In [2]:
"""
EquiGuard Data Pipeline

Prepares training, validation, and test datasets for
equine lameness detection using IMU-derived gait features.

Datasets:
- Figshare B
- EQUIMOVES (Gmel 2024)
"""

'\nEquiGuard Data Pipeline\n\nPrepares training, validation, and test datasets for\nequine lameness detection using IMU-derived gait features.\n\nDatasets:\n- Figshare B\n- EQUIMOVES (Gmel 2024)\n'

In [3]:
# Paths and constants
import os
import numpy as np
import pandas as pd
from sklearn.utils import shuffle

np.random.seed(42)

RAW_DIR = r"C:\Project_EquiGuard\data\raw"
PROCESSED_DIR = r"C:\Project_EquiGuard\data\processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

FIGSHARE_FILE = os.path.join(
    RAW_DIR,
    "Results Spreadsheets for figshare with age and mass (1).xlsx"
)

HORSE_ID_COL = "horseID"
HEAD_THRESHOLD = 6
PELVIS_THRESHOLD = 3
# Clinical thresholds from Keegan et al. (2007)

print("Setup complete.")
print("Figshare exists:", os.path.exists(FIGSHARE_FILE))

Setup complete.
Figshare exists: True


In [4]:
# Load and inspect Figshare B

figshare = pd.read_excel(FIGSHARE_FILE)

print(f"Shape: {figshare.shape}")
print(f"Columns: {list(figshare.columns)}")
print(f"\nNull values:\n{figshare.isnull().sum()}")

Shape: (307, 34)
Columns: ['horseID', 'STUDY', 'Age (years)', 'Mass (Kg)', 'MnDSac', 'MxDSac', 'UpDSac', 'HHD', 'MnDHead', 'MxDHead', 'UpDHead', 'MnDisLHasym', 'MnDisRHasym', 'MnDisHsym', 'MxDisLHasym', 'MxDisRHasym', 'MxDisHsym', 'UpDisLHasym', 'UpDisRHasym', 'UpDisHsym', 'HHDisLHasym', 'HHDisRHasym', 'HHDisHsym', 'MnDisLFasym', 'MnDisRFasym', 'MnDisFsym', 'MxDisLFasym', 'MxDisRFasym', 'MxDisFsym', 'UpDisLFasym', 'UpDisRFasym', 'UpDisFsym', 'Max value', 'Exclude']

Null values:
horseID        0
STUDY          0
Age (years)    0
Mass (Kg)      2
MnDSac         0
MxDSac         0
UpDSac         0
HHD            0
MnDHead        0
MxDHead        0
UpDHead        0
MnDisLHasym    0
MnDisRHasym    0
MnDisHsym      0
MxDisLHasym    0
MxDisRHasym    0
MxDisHsym      0
UpDisLHasym    0
UpDisRHasym    0
UpDisHsym      0
HHDisLHasym    0
HHDisRHasym    0
HHDisHsym      0
MnDisLFasym    0
MnDisRFasym    0
MnDisFsym      0
MxDisLFasym    0
MxDisRFasym    0
MxDisFsym      0
UpDisLFasym    0
UpDisR

In [5]:
# # Generate lameness labels
rows_before = len(figshare)
figshare = figshare[figshare["Exclude"] == 0].reset_index(drop=True)
print(f"Rows removed (Exclude flag): {rows_before - len(figshare)}")
print(f"Rows remaining: {len(figshare)}")

head_asym = figshare["MnDHead"].abs()
pelvis_asym = figshare["MnDSac"].abs()

figshare["label"] = np.where(
    (head_asym > HEAD_THRESHOLD) | (pelvis_asym > PELVIS_THRESHOLD),
    1, 0
)

print(f"\nLabel distribution:")
print(figshare["label"].value_counts())
print(f"Lame %: {figshare['label'].mean()*100:.1f}%")

figshare.to_csv(os.path.join(PROCESSED_DIR, "figshare_b_labeled.csv"), index=False)
print("\nSaved: figshare_b_labeled.csv")

Rows removed (Exclude flag): 0
Rows remaining: 307

Label distribution:
label
1    245
0     62
Name: count, dtype: int64
Lame %: 79.8%

Saved: figshare_b_labeled.csv


In [6]:
# Horse-aware train/validation/test split

horse_ids = figshare[HORSE_ID_COL].unique()
print(f"Total unique horses: {len(horse_ids)}")

# Separate sound and lame horse IDs
sound_ids = figshare[figshare["label"] == 0][HORSE_ID_COL].unique()
lame_ids  = figshare[figshare["label"] == 1][HORSE_ID_COL].unique()

print(f"Sound horses: {len(sound_ids)}")
print(f"Lame horses:  {len(lame_ids)}")

# Keep sound horses for training
# Use lame horses for validation and test evaluation
np.random.seed(42)
np.random.shuffle(lame_ids)

split_index = len(lame_ids) // 2
val_ids  = lame_ids[:split_index]
test_ids = lame_ids[split_index:]

# Add some sound horses to val and test for evaluation realism
# Take 10 sound horses for val, 10 for test, keep rest for training
np.random.seed(42)
np.random.shuffle(sound_ids)

val_sound_ids   = sound_ids[:10]
test_sound_ids  = sound_ids[10:20]
train_sound_ids = sound_ids[20:]  # remaining sound horses for training

# Final val and test IDs
val_ids_final  = list(val_ids)  + list(val_sound_ids)
test_ids_final = list(test_ids) + list(test_sound_ids)

val_df  = figshare[figshare[HORSE_ID_COL].isin(val_ids_final)]
test_df = figshare[figshare[HORSE_ID_COL].isin(test_ids_final)]

print(f"\nVal rows: {len(val_df)}")
print(f"Val labels:\n{val_df['label'].value_counts()}")

print(f"\nTest rows: {len(test_df)}")
print(f"Test labels:\n{test_df['label'].value_counts()}")

print(f"\nSound horses reserved for training: {len(train_sound_ids)}")

val_df.to_csv(os.path.join(PROCESSED_DIR, "val_raw.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "test_raw.csv"), index=False)
print("\nSaved: val_raw.csv, test_raw.csv")

Total unique horses: 307
Sound horses: 62
Lame horses:  245

Val rows: 132
Val labels:
label
1    122
0     10
Name: count, dtype: int64

Test rows: 133
Test labels:
label
1    123
0     10
Name: count, dtype: int64

Sound horses reserved for training: 42

Saved: val_raw.csv, test_raw.csv


In [7]:
# Generate synthetic lame samples

val_test_ids = set(val_ids_final + test_ids_final)

sound_horses = figshare[
    figshare[HORSE_ID_COL].isin(train_sound_ids)
].copy()

print(f"Sound horses available for training: {len(sound_horses)}")

np.random.seed(42)

variants = []
N_VARIANTS = 4

for i in range(N_VARIANTS):

    synthetic = sound_horses.copy()

    # Convert to float so noise can be added safely
    synthetic["MnDHead"] = synthetic["MnDHead"].astype(float)
    synthetic["MnDSac"]  = synthetic["MnDSac"].astype(float)

    n = len(synthetic)

    # Randomly assign forelimb or hindlimb lameness
    mask = np.random.rand(n) > 0.5

    # Forelimb lame: increase head asymmetry
    shift_head = np.random.uniform(7, 15, n)
    noise_head = np.random.normal(0, 1.5, n)

    synthetic.loc[mask, "MnDHead"] += (
        shift_head[mask] + noise_head[mask]
    )

    # Hindlimb lame: increase pelvis asymmetry
    shift_sac = np.random.uniform(4, 10, n)
    noise_sac = np.random.normal(0, 1.0, n)

    synthetic.loc[~mask, "MnDSac"] += (
        shift_sac[~mask] + noise_sac[~mask]
    )

    synthetic["MnDHead"] = synthetic["MnDHead"].round(2)
    synthetic["MnDSac"]  = synthetic["MnDSac"].round(2)

    synthetic["label"] = 1
    synthetic["source"] = f"synthetic_lame_v{i+1}"

    variants.append(synthetic)

# Combine all synthetic variants
all_synthetic = pd.concat(variants, ignore_index=True)

# Label original sound horses
sound_horses = sound_horses.copy()
sound_horses["source"] = "real_sound"

# Final training dataset
train_data = pd.concat(
    [sound_horses, all_synthetic],
    ignore_index=True
)

print(f"\nTraining set shape: {train_data.shape}")

print(f"\nLabel distribution:")
print(train_data["label"].value_counts())

print(f"\nSource distribution:")
print(train_data["source"].value_counts())

# Leakage check
train_horse_ids = set(train_data[HORSE_ID_COL].unique())
leakage = train_horse_ids.intersection(val_test_ids)

print(f"\nLeakage check (should be 0): {len(leakage)}")

# Save dataset
train_data.to_csv(
    os.path.join(PROCESSED_DIR, "train_raw.csv"),
    index=False
)

print("\nSaved: train_raw.csv")

Sound horses available for training: 42

Training set shape: (210, 36)

Label distribution:
label
1    168
0     42
Name: count, dtype: int64

Source distribution:
source
real_sound           42
synthetic_lame_v1    42
synthetic_lame_v2    42
synthetic_lame_v3    42
synthetic_lame_v4    42
Name: count, dtype: int64

Leakage check (should be 0): 0

Saved: train_raw.csv


In [8]:
# Dataset verification

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_raw.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val_raw.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test_raw.csv"))

print("TRAINING SET")
print(f"Rows: {len(train_df)}")
print(f"Labels:\n{train_df['label'].value_counts()}")

print("\nVALIDATION SET")
print(f"Rows: {len(val_df)}")
print(f"Labels:\n{val_df['label'].value_counts()}")

print("\nTEST SET")
print(f"Rows: {len(test_df)}")
print(f"Labels:\n{test_df['label'].value_counts()}")

# Horse overlap check
val_horses   = set(val_df[HORSE_ID_COL])
test_horses  = set(test_df[HORSE_ID_COL])
train_horses = set(train_df[HORSE_ID_COL])

print(f"\nVal/Test overlap (should be 0): {len(val_horses & test_horses)}")
print(f"Train/Val overlap (should be 0): {len(train_horses & val_horses)}")
print(f"Train/Test overlap (should be 0): {len(train_horses & test_horses)}")

print("\nPipeline complete.")


TRAINING SET
Rows: 210
Labels:
label
1    168
0     42
Name: count, dtype: int64

VALIDATION SET
Rows: 132
Labels:
label
1    122
0     10
Name: count, dtype: int64

TEST SET
Rows: 133
Labels:
label
1    123
0     10
Name: count, dtype: int64

Val/Test overlap (should be 0): 0
Train/Val overlap (should be 0): 0
Train/Test overlap (should be 0): 0

Pipeline complete.
